In [1]:
from pathlib import Path
import re
import pydicom
from pydicom.uid import ExplicitVRLittleEndian
import numpy as np
from PIL import Image
from pydicom.uid import JPEG2000Lossless

In [2]:
HD21_SCAN_WIDTH_MM = 9.0  # The width of the HD21 scan in millimeters, for pixel spacing metadata
INPUT_DIR = Path("../AutoExport_Raj/CZMI1260567209")
BASE_DICOM_PATH = Path("./zeiss_export/DATAFILES/E584/1BO5BKDBYO0B857TZSUQ3LBV9D4KKGEOM27IOLS4X2ZU.EX.DCM")
COMPRESS_OUTPUT = False

In [3]:
ds = pydicom.dcmread(BASE_DICOM_PATH)

In [4]:
# 1. Define your input directory with a broad glob
file_paths = list(INPUT_DIR.glob("**/*.tiff"))

hd21_files_grouped = {}
for filepath in file_paths:
    match = re.search(r'HD 21 Line_(\d+)_', filepath.name)
    if match:
        datetime_key = match.group(1)
        if not datetime_key in hd21_files_grouped:
            hd21_files_grouped[datetime_key] = []
        hd21_files_grouped[datetime_key].append(filepath)

# Sort numerically, by line #, rather than alphabetically
def extract_line_number(filepath):
    match = re.search(r'line(\d+)\.tif', filepath.name)
    if match:
        return int(match.group(1))
    return -1 # Fallback in case a file doesn't match perfectly
for datetime_key in hd21_files_grouped:
    hd21_files_grouped[datetime_key].sort(key=extract_line_number)
    assert len(hd21_files_grouped[datetime_key]) == 21, f"Expected 21 files for {datetime_key}, but found {len(hd21_files_grouped[datetime_key])}."

print(f"Number of HD 21 file groups: {len(hd21_files_grouped)}")

Number of HD 21 file groups: 2


In [5]:
file_group = list(hd21_files_grouped.values())[0] # use first group for demo
frames = []
for file in file_group:
    # Open, convert to 8-bit grayscale, and cast to uint8 array
    pixel_array = np.array(Image.open(file).convert('L'), dtype=np.uint8)
    frames.append(pixel_array)

# Merge the list of 2D arrays into a 3D array: shape becomes (21, Rows, Columns)
volume_array = np.stack(frames, dtype=np.uint8)

In [6]:
# Set basic frame and spatial dimensions
ds.NumberOfFrames = volume_array.shape[0]
ds.Rows = volume_array.shape[1]
ds.Columns = volume_array.shape[2]
# Infer pixel bit depth dynamically from numpy dtype
assert volume_array.dtype == np.uint8
ds.SamplesPerPixel = 1
ds.BitsAllocated = volume_array.itemsize * 8
ds.BitsStored = ds.BitsAllocated
ds.HighBit = ds.BitsAllocated - 1
ds.PixelRepresentation = 0  # 0 = Unsigned, 1 = Signed

ds.PhotometricInterpretation = "MONOCHROME2"

# Update Pixel Spacing
pixel_spacing = HD21_SCAN_WIDTH_MM / ds.Columns
ds.PixelSpacing = [pixel_spacing, pixel_spacing] # isotropic image

# Remove old Windowing tags so the viewer auto-windows the image
if "WindowCenter" in ds:
    del ds.WindowCenter
if "WindowWidth" in ds:
    del ds.WindowWidth

# Set endianness and VR explicit mode
ds.is_little_endian = True
ds.is_implicit_VR = False

if COMPRESS_OUTPUT:
    # Encodes pixels and automatically updates ds.PixelData & ds.file_meta.TransferSyntaxUID
    ds.compress(JPEG2000Lossless, volume_array)
else:
    # Explicit VR Little Endian Transfer Syntax UID for uncompressed raw data
    ds.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    ds.PixelData = volume_array.tobytes()

/Users/raj/code/miniconda3/envs/imind_cnn_app/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (46) exceeds the maximum length of 16 allowed for VR SH.
  warn_and_log(msg)


In [7]:
output_filepath = Path(f"hd21_{'compressed' if COMPRESS_OUTPUT else 'uncompressed'}.dcm")
ds.save_as(output_filepath)
print(f"DICOM file successfully overwritten with {ds.NumberOfFrames} frames!")

DICOM file successfully overwritten with 21 frames!
